# EDA: Generated Interactions and Products

This notebook explores the synthetic interactions and product catalog to inform recommender design.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

profiles = pd.read_csv(os.path.join(root, 'generated_data', 'profiles.csv'))
interactions = pd.read_csv(os.path.join(root, 'generated_data', 'product_interactions.csv'))
products = pd.read_csv(os.path.join(root, 'collected_data', 'products_rows.csv'))

print('Rows:', { 'profiles': len(profiles), 'interactions': len(interactions), 'products': len(products) })

interactions['created_at'] = pd.to_datetime(interactions['created_at'], errors='coerce')

# Event mix
plt.figure(figsize=(6,3))
interactions['interaction_type'].value_counts().plot(kind='bar'); plt.title('Event Mix'); plt.tight_layout(); plt.show()

# Per-user histogram
plt.figure(figsize=(6,3))
interactions.groupby('user_id').size().plot(kind='hist', bins=30); plt.title('Interactions per User'); plt.tight_layout(); plt.show()

# Per-item histogram
plt.figure(figsize=(6,3))
interactions.groupby('product_id').size().plot(kind='hist', bins=30); plt.title('Interactions per Item'); plt.tight_layout(); plt.show()


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Sparsity level
num_users = interactions['user_id'].nunique()
num_items = interactions['product_id'].nunique()
num_events = len(interactions)
sparsity = 1.0 - (num_events / (num_users * num_items + 1e-9))
print({'num_users': num_users, 'num_items': num_items, 'events': num_events, 'sparsity': sparsity})

# Top categories if present
if 'category_id' in products.columns:
    print(products['category_id'].value_counts().head(10))

# Cold-start ratios
train_cut = interactions['created_at'].quantile(0.7)
train = interactions[interactions['created_at'] <= train_cut]
val = interactions[interactions['created_at'] > train_cut]
seen_items = set(train['product_id'].unique())
seen_users = set(train['user_id'].unique())
cold_items_ratio = 1.0 - (val['product_id'].isin(seen_items).mean())
cold_users_ratio = 1.0 - (val['user_id'].isin(seen_users).mean())
print({'cold_items_ratio': cold_items_ratio, 'cold_users_ratio': cold_users_ratio})


# EDA: Generated implicit data and products


In [ ]:
import os, pandas as pd
root = os.path.dirname(os.path.dirname(os.path.abspath('.')))
gd = os.path.join(root, 'generated_data')
cd = os.path.join(root, 'collected_data')
profiles = pd.read_csv(os.path.join(gd, 'profiles.csv'))
interactions = pd.read_csv(os.path.join(gd, 'product_interactions.csv'))
orders = pd.read_csv(os.path.join(gd, 'orders.csv'))
order_items = pd.read_csv(os.path.join(gd, 'order_items.csv'))
products = pd.read_csv(os.path.join(cd, 'products_rows.csv'))
variants = pd.read_csv(os.path.join(cd, 'product_variants_rows.csv'))
profiles.head(), interactions.head(), products.head()


In [ ]:
def summary(df):
    return pd.DataFrame({
        'rows': [len(df)],
        'n_cols': [df.shape[1]],
        'null_pct_mean': [df.isnull().mean().mean()],
    })
summary(profiles), summary(interactions), summary(products)


In [ ]:
interactions['created_at'] = pd.to_datetime(interactions['created_at'], errors='coerce')
time_span = interactions['created_at'].min(), interactions['created_at'].max()
event_counts = interactions['interaction_type'].value_counts()
per_user = interactions.groupby('user_id').size().describe()
per_item = interactions.groupby('product_id').size().describe()
time_span, event_counts, per_user, per_item


In [ ]:
products['is_active'] = products['is_active'].astype(str).str.lower()
active = products[products['is_active']=='true']
active['inventory_quantity'] = pd.to_numeric(active['inventory_quantity'], errors='coerce').fillna(0).astype(int)
active.describe(include='all')
